# Flow Matching from Scratch

**Lecture 3, Part 1 — Vizuara Modern Robot Learning Bootcamp**

We build **Flow Matching** from first principles, starting with a DDPM recap and ending with conditioned robot action generation.

| Section | What you'll build | Key idea |
|---------|------------------|----------|
| Step 1 | Minimal DDPM | Forward noise + reverse denoising (recap) |
| Step 2 | DDPM's two problems | Slow inference (1000 steps) + curved paths |
| Step 3 | Linear interpolation | The simplest path: straight lines |
| Step 4 | Velocity network | Predict direction, not noise |
| Step 5 | Flow matching training | 15-line training loop |
| Step 6 | Euler integration | Generate samples in 10 steps (not 1000!) |
| Step 7 | OT coupling | Optimal transport = uncrossed paths |
| Step 8 | Robot actions | Conditioned generation for 4 push directions |
| Step 9 | Head-to-head | DDPM vs Flow Matching comparison |

**Running example:** 2D Swiss roll (a curved manifold) that we'll later swap for robot action trajectories.

---

In [ ]:
!pip install -q torch numpy matplotlib scipy scikit-learn 2>&1 | tail -3

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time
import math
from sklearn.datasets import make_swiss_roll
from scipy.optimize import linear_sum_assignment

# Vizuara color palette
ACCENT = '#D97757'
BLUE   = '#5B8CB8'
TEAL   = '#7DA488'
PURPLE = '#9B7EC8'
WARM   = '#C4956A'
BG     = '#FAF8F5'

plt.rcParams['figure.facecolor'] = BG
plt.rcParams['axes.facecolor']   = BG
plt.rcParams['font.size']        = 12

torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
# ── Swiss roll dataset ──────────────────────────────────────────
def get_swiss_roll(n_samples=2000):
    """Generate 2D Swiss roll data, normalized to [-2, 2]."""
    data_3d, _ = make_swiss_roll(n_samples=n_samples, noise=0.5)
    data_2d = data_3d[:, [0, 2]]  # take X and Z columns
    # Normalize to [-2, 2]
    data_2d = data_2d - data_2d.mean(axis=0)
    data_2d = data_2d / (data_2d.std(axis=0) + 1e-8) * 1.5
    return torch.tensor(data_2d, dtype=torch.float32)

data = get_swiss_roll(2000)
print(f'Swiss roll data shape: {list(data.shape)}')
print(f'  Range: [{data.min():.2f}, {data.max():.2f}]')

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(data[:, 0].numpy(), data[:, 1].numpy(), s=4, alpha=0.5, c=ACCENT)
ax.set_title('Swiss Roll — Our Target Distribution', fontweight='bold', color=ACCENT, fontsize=14)
ax.set_xlabel('x', fontsize=11)
ax.set_ylabel('y', fontsize=11)
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()
print('Goal: learn to generate NEW points that look like this distribution.')

---
## Step 1: DDPM Recap — Forward Noise + Reverse Denoising

Before building flow matching, let's recap **DDPM** (Denoising Diffusion Probabilistic Models):

1. **Forward process:** Gradually add Gaussian noise to data over T timesteps  
   $x_t = \sqrt{\bar{\alpha}_t} \, x_0 + \sqrt{1 - \bar{\alpha}_t} \, \epsilon$

2. **Reverse process:** Learn to predict the noise $\epsilon$ at each step, then subtract it

3. **Noise schedule:** $\beta_t$ controls how much noise is added at each step  
   $\alpha_t = 1 - \beta_t$, $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$

The key insight: at $t=T$, data is pure noise. We train a network to reverse this process step by step.

In [ ]:
class SimpleDDPM:
    """Minimal DDPM: forward noise + reverse denoising schedule."""
    def __init__(self, T=100, beta_start=1e-4, beta_end=0.02):
        self.T = T
        self.betas = torch.linspace(beta_start, beta_end, T)
        self.alphas = 1.0 - self.betas
        self.alpha_bar = torch.cumprod(self.alphas, dim=0)
        # Precompute coefficients
        self.sqrt_alpha_bar = torch.sqrt(self.alpha_bar)
        self.sqrt_one_minus_alpha_bar = torch.sqrt(1.0 - self.alpha_bar)

    def forward_diffusion(self, x0, t):
        """Add noise to x0 at timestep t: x_t = sqrt(alpha_bar_t)*x0 + sqrt(1-alpha_bar_t)*eps."""
        eps = torch.randn_like(x0)
        sqrt_ab = self.sqrt_alpha_bar[t].unsqueeze(-1)           # [batch, 1]
        sqrt_omab = self.sqrt_one_minus_alpha_bar[t].unsqueeze(-1)
        x_t = sqrt_ab * x0 + sqrt_omab * eps
        return x_t, eps

    def reverse_step(self, x_t, eps_pred, t):
        """One DDPM reverse step: remove predicted noise."""
        alpha_t = self.alphas[t]
        alpha_bar_t = self.alpha_bar[t]
        beta_t = self.betas[t]
        # Mean of p(x_{t-1} | x_t)
        coef1 = 1.0 / torch.sqrt(alpha_t)
        coef2 = beta_t / torch.sqrt(1.0 - alpha_bar_t)
        mean = coef1 * (x_t - coef2 * eps_pred)
        # Add noise (except at t=0)
        if t > 0:
            sigma = torch.sqrt(beta_t)
            mean = mean + sigma * torch.randn_like(x_t)
        return mean


ddpm = SimpleDDPM(T=100, beta_start=1e-4, beta_end=0.02)
print(f'DDPM with T={ddpm.T} steps')
print(f'  beta range: [{ddpm.betas[0]:.4f}, {ddpm.betas[-1]:.4f}]')
print(f'  alpha_bar[0] = {ddpm.alpha_bar[0]:.4f} (almost no noise)')
print(f'  alpha_bar[-1] = {ddpm.alpha_bar[-1]:.4f} (almost pure noise)')

In [ ]:
# Visualize the forward diffusion process on Swiss roll
timesteps_to_show = [0, 10, 25, 50, 75, 99]
fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(22, 4))

for i, t in enumerate(timesteps_to_show):
    t_batch = torch.full((data.shape[0],), t, dtype=torch.long)
    x_t, _ = ddpm.forward_diffusion(data, t_batch)
    ax = axes[i]
    ax.scatter(x_t[:, 0].numpy(), x_t[:, 1].numpy(), s=3, alpha=0.4, c=ACCENT)
    ax.set_title(f't = {t}\n$\\bar{{\\alpha}}$={ddpm.alpha_bar[t]:.3f}',
                 fontweight='bold', color=ACCENT, fontsize=11)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)

plt.suptitle('DDPM Forward Process: Data → Noise', fontsize=15,
             fontweight='bold', color=ACCENT)
plt.tight_layout()
plt.show()
print('At t=0: pure data. At t=99: nearly pure Gaussian noise.')
print('The forward process destroys structure gradually.')

In [ ]:
# Plot the noise schedule: alpha_bar over time
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(range(ddpm.T), ddpm.alpha_bar.numpy(), color=ACCENT, linewidth=2.5)
ax.set_xlabel('Timestep t', fontsize=11)
ax.set_ylabel('$\\bar{\\alpha}_t$', fontsize=13)
ax.set_title('Cumulative Signal Retention $\\bar{\\alpha}_t$',
             fontweight='bold', color=ACCENT, fontsize=13)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4)
ax.text(50, 0.52, '50% signal, 50% noise', fontsize=9, color='gray')
ax.grid(True, alpha=0.2)

ax = axes[1]
ax.plot(range(ddpm.T), ddpm.betas.numpy(), color=BLUE, linewidth=2.5, label='$\\beta_t$')
ax.plot(range(ddpm.T), ddpm.alphas.numpy(), color=TEAL, linewidth=2.5, label='$\\alpha_t$')
ax.set_xlabel('Timestep t', fontsize=11)
ax.set_ylabel('Value', fontsize=13)
ax.set_title('Per-Step Schedule: $\\beta_t$ and $\\alpha_t$',
             fontweight='bold', color=BLUE, fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()
print('DDPM requires carefully designed noise schedules.')
print('Too fast → data destroyed too quickly. Too slow → 1000+ steps needed.')
print('Flow matching eliminates this complexity entirely.')

---
## Step 2: The Two Problems with DDPM

DDPM works, but it has two major issues:

1. **Slow inference:** Needs many steps (50-1000) to generate a sample. Each step requires a full neural network forward pass.

2. **Curved, stochastic paths:** The reverse process follows curved, noisy trajectories from noise to data. This makes it hard to take shortcuts (fewer steps = worse quality).

Let's see both problems in action.

In [ ]:
# ── Train a tiny DDPM noise predictor on Swiss roll ─────────────
class DDPMNoiseNet(nn.Module):
    """Predict noise epsilon given (x_t, t). MLP with sinusoidal time embedding."""
    def __init__(self, data_dim=2, hidden=256, time_dim=64):
        super().__init__()
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, hidden),
            nn.ReLU(),
        )
        self.net = nn.Sequential(
            nn.Linear(data_dim + hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, data_dim),
        )
        self.time_dim = time_dim

    def sinusoidal_embed(self, t, dim):
        """Sinusoidal time embedding: t -> [sin, cos] features."""
        half = dim // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device).float() / half)
        args = t.float().unsqueeze(-1) * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

    def forward(self, x, t):
        # t: [batch] integer timestep
        t_emb = self.sinusoidal_embed(t, self.time_dim)  # [batch, time_dim]
        t_feat = self.time_mlp(t_emb)                     # [batch, hidden]
        h = torch.cat([x, t_feat], dim=-1)                # [batch, data_dim + hidden]
        return self.net(h)


# Train DDPM
ddpm_net = DDPMNoiseNet(data_dim=2, hidden=256).to(device)
optimizer = torch.optim.Adam(ddpm_net.parameters(), lr=1e-3)
data_device = data.to(device)

ddpm_losses = []
ddpm_net.train()
for step in range(2000):
    idx = torch.randint(0, len(data), (256,))
    x0 = data_device[idx]
    t = torch.randint(0, ddpm.T, (256,), device=device)

    # Forward diffusion (on CPU for alpha_bar, move result)
    eps = torch.randn_like(x0)
    sqrt_ab = ddpm.sqrt_alpha_bar[t.cpu()].unsqueeze(-1).to(device)
    sqrt_omab = ddpm.sqrt_one_minus_alpha_bar[t.cpu()].unsqueeze(-1).to(device)
    x_t = sqrt_ab * x0 + sqrt_omab * eps

    # Predict noise
    eps_pred = ddpm_net(x_t, t)
    loss = F.mse_loss(eps_pred, eps)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    ddpm_losses.append(loss.item())

    if step % 500 == 0:
        print(f'  Step {step:4d}: loss = {loss.item():.4f}')

print(f'\nDDPM training done. Final loss: {ddpm_losses[-1]:.4f}')

In [ ]:
# Problem 1: Visualize DDPM's curved, stochastic reverse paths
@torch.no_grad()
def ddpm_sample_with_paths(ddpm, model, n_samples=5, T=100):
    """Run DDPM reverse process, recording full trajectories."""
    model.eval()
    x = torch.randn(n_samples, 2, device=device)
    trajectory = [x.cpu().numpy().copy()]

    for t in reversed(range(T)):
        t_batch = torch.full((n_samples,), t, dtype=torch.long, device=device)
        eps_pred = model(x, t_batch)

        alpha_t = ddpm.alphas[t].to(device)
        alpha_bar_t = ddpm.alpha_bar[t].to(device)
        beta_t = ddpm.betas[t].to(device)

        coef1 = 1.0 / torch.sqrt(alpha_t)
        coef2 = beta_t / torch.sqrt(1.0 - alpha_bar_t)
        mean = coef1 * (x - coef2 * eps_pred)

        if t > 0:
            sigma = torch.sqrt(beta_t)
            mean = mean + sigma * torch.randn_like(x)
        x = mean
        trajectory.append(x.cpu().numpy().copy())

    return np.array(trajectory)  # [T+1, n_samples, 2]


paths_ddpm = ddpm_sample_with_paths(ddpm, ddpm_net, n_samples=5, T=100)
print(f'DDPM paths shape: {paths_ddpm.shape}  — {paths_ddpm.shape[0]} steps, {paths_ddpm.shape[1]} samples')

fig, ax = plt.subplots(figsize=(7, 7))
colors = [ACCENT, BLUE, TEAL, PURPLE, WARM]
for i in range(5):
    path = paths_ddpm[:, i, :]  # [T+1, 2]
    ax.plot(path[:, 0], path[:, 1], '-', color=colors[i], alpha=0.6, linewidth=1.2)
    ax.scatter(path[0, 0], path[0, 1], marker='o', s=60, color=colors[i], zorder=5,
               edgecolors='white', linewidths=1)
    ax.scatter(path[-1, 0], path[-1, 1], marker='*', s=120, color=colors[i], zorder=5,
               edgecolors='white', linewidths=1)

ax.scatter(data[:500, 0].numpy(), data[:500, 1].numpy(), s=2, alpha=0.15, c='gray')
ax.set_title('DDPM Reverse Paths: Noise → Data (100 steps)',
             fontweight='bold', color=ACCENT, fontsize=14)
ax.set_xlabel('x', fontsize=11)
ax.set_ylabel('y', fontsize=11)
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)
ax.legend(['Path 1', 'Path 2', 'Path 3', 'Path 4', 'Path 5'],
          loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()
print('Notice: paths are CURVED and STOCHASTIC (wiggly).')
print('Each step adds random noise, so paths meander.')
print('This makes it hard to take shortcuts with fewer steps.')

In [ ]:
# Problem 2: Inference speed — time DDPM at different step counts
@torch.no_grad()
def ddpm_sample(ddpm_obj, model, n_samples=500, T=100):
    model.eval()
    x = torch.randn(n_samples, 2, device=device)
    for t in reversed(range(T)):
        t_batch = torch.full((n_samples,), t, dtype=torch.long, device=device)
        eps_pred = model(x, t_batch)
        alpha_t = ddpm_obj.alphas[t].to(device)
        alpha_bar_t = ddpm_obj.alpha_bar[t].to(device)
        beta_t = ddpm_obj.betas[t].to(device)
        coef1 = 1.0 / torch.sqrt(alpha_t)
        coef2 = beta_t / torch.sqrt(1.0 - alpha_bar_t)
        mean = coef1 * (x - coef2 * eps_pred)
        if t > 0:
            mean = mean + torch.sqrt(beta_t) * torch.randn_like(x)
        x = mean
    return x


# Build DDPM variants at different T values
step_counts = [50, 100, 500, 1000]
ddpm_times = []

for T in step_counts:
    ddpm_var = SimpleDDPM(T=T)
    # Warm up
    _ = ddpm_sample(ddpm_var, ddpm_net, n_samples=100, T=min(T, ddpm.T))
    # Time it
    start = time.perf_counter()
    for _ in range(3):
        _ = ddpm_sample(ddpm_var, ddpm_net, n_samples=500, T=min(T, ddpm.T))
    elapsed = (time.perf_counter() - start) / 3
    ddpm_times.append(elapsed * 1000)  # ms
    print(f'  T={T:4d}: {elapsed*1000:.1f} ms')

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar([str(s) for s in step_counts], ddpm_times, color=[ACCENT, BLUE, TEAL, PURPLE],
              edgecolor='white', linewidth=1.5)
ax.set_xlabel('Number of Denoising Steps (T)', fontsize=12)
ax.set_ylabel('Inference Time (ms)', fontsize=12)
ax.set_title('DDPM Inference Time: More Steps = Slower',
             fontweight='bold', color=ACCENT, fontsize=14)
for bar, t_ms in zip(bars, ddpm_times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{t_ms:.0f} ms', ha='center', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.show()
print('For a robot running at 10 Hz, you have 100 ms per action.')
print('DDPM with 1000 steps is FAR too slow for real-time control.')
print('\nFlow matching solves this: good samples in just 5-10 steps!')

---
## Step 3: Linear Interpolation — The Simplest Path

Flow matching's key insight: **why curve when you can go straight?**

Instead of DDPM's complex noise schedule, define a simple linear path:

$$x_\tau = (1 - \tau) \, x_0 + \tau \, x_1$$

where:
- $x_0 \sim \mathcal{N}(0, I)$ — start from noise
- $x_1$ — a real data point (target)
- $\tau \in [0, 1]$ — interpolation parameter (0 = noise, 1 = data)

The path from $x_0$ to $x_1$ is a **straight line**. No schedule needed. No stochasticity.

In [ ]:
# Linear interpolation between noise and data
n_paths = 8
x0_noise = torch.randn(n_paths, 2)         # noise samples
idx = torch.randint(0, len(data), (n_paths,))
x1_data = data[idx]                          # data samples

# Interpolate
n_interp = 50
taus = torch.linspace(0, 1, n_interp)
interp_paths = []  # [n_paths, n_interp, 2]
for i in range(n_paths):
    path = []
    for tau in taus:
        x_tau = (1 - tau) * x0_noise[i] + tau * x1_data[i]
        path.append(x_tau.numpy())
    interp_paths.append(np.array(path))

print(f'Generated {n_paths} straight-line paths from noise to data')
print(f'  x0 (noise):  mean={x0_noise.mean():.2f}, std={x0_noise.std():.2f}')
print(f'  x1 (data):   mean={x1_data.mean():.2f}, std={x1_data.std():.2f}')

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(data[:500, 0].numpy(), data[:500, 1].numpy(), s=2, alpha=0.15, c='gray',
           label='Data')
path_colors = [ACCENT, BLUE, TEAL, PURPLE, WARM, '#E07B5A', '#6BA87D', '#7B8EC8']
for i in range(n_paths):
    p = interp_paths[i]
    ax.plot(p[:, 0], p[:, 1], '-', color=path_colors[i % len(path_colors)],
            linewidth=1.5, alpha=0.7)
    ax.scatter(p[0, 0], p[0, 1], marker='o', s=50, color=path_colors[i % len(path_colors)],
               edgecolors='white', linewidths=1, zorder=5)
    ax.scatter(p[-1, 0], p[-1, 1], marker='*', s=100, color=path_colors[i % len(path_colors)],
               edgecolors='white', linewidths=1, zorder=5)

ax.set_title('Flow Matching: Straight-Line Paths (Noise → Data)',
             fontweight='bold', color=TEAL, fontsize=14)
ax.set_xlabel('x', fontsize=11)
ax.set_ylabel('y', fontsize=11)
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)
ax.annotate('circles = noise (x0)', xy=(0.02, 0.95), xycoords='axes fraction',
            fontsize=10, color='gray')
ax.annotate('stars = data (x1)', xy=(0.02, 0.90), xycoords='axes fraction',
            fontsize=10, color='gray')
plt.tight_layout()
plt.show()
print('Each path is a STRAIGHT LINE from noise to data.')
print('No noise schedule, no stochasticity, no curved trajectories.')

In [ ]:
# Side-by-side comparison: DDPM curved paths vs Flow Matching straight paths
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Left: DDPM paths (reuse from earlier)
ax = axes[0]
ax.scatter(data[:500, 0].numpy(), data[:500, 1].numpy(), s=2, alpha=0.15, c='gray')
for i in range(5):
    path = paths_ddpm[:, i, :]
    ax.plot(path[:, 0], path[:, 1], '-', color=colors[i], alpha=0.6, linewidth=1.2)
    ax.scatter(path[0, 0], path[0, 1], marker='o', s=60, color=colors[i], zorder=5,
               edgecolors='white', linewidths=1)
    ax.scatter(path[-1, 0], path[-1, 1], marker='*', s=120, color=colors[i], zorder=5,
               edgecolors='white', linewidths=1)
ax.set_title('DDPM: Curved, Stochastic Paths', fontweight='bold', color=ACCENT, fontsize=14)
ax.set_xlabel('x', fontsize=11)
ax.set_ylabel('y', fontsize=11)
ax.set_aspect('equal')
ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.grid(True, alpha=0.2)

# Right: Flow Matching paths
ax = axes[1]
ax.scatter(data[:500, 0].numpy(), data[:500, 1].numpy(), s=2, alpha=0.15, c='gray')
for i in range(min(5, n_paths)):
    p = interp_paths[i]
    ax.plot(p[:, 0], p[:, 1], '-', color=colors[i], alpha=0.7, linewidth=1.5)
    ax.scatter(p[0, 0], p[0, 1], marker='o', s=60, color=colors[i], zorder=5,
               edgecolors='white', linewidths=1)
    ax.scatter(p[-1, 0], p[-1, 1], marker='*', s=120, color=colors[i], zorder=5,
               edgecolors='white', linewidths=1)
ax.set_title('Flow Matching: Straight Paths', fontweight='bold', color=TEAL, fontsize=14)
ax.set_xlabel('x', fontsize=11)
ax.set_ylabel('y', fontsize=11)
ax.set_aspect('equal')
ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.grid(True, alpha=0.2)

plt.suptitle('DDPM vs Flow Matching: Path Geometry',
             fontsize=16, fontweight='bold', color=ACCENT)
plt.tight_layout()
plt.show()
print('Left (DDPM): Paths curve and wander — hard to take shortcuts.')
print('Right (FM):  Paths are straight lines — easy to take large steps.')
print('\nThis is WHY flow matching needs fewer steps for good quality.')

---
## Step 4: The Velocity Field

Instead of predicting **noise** $\epsilon$ (like DDPM), flow matching predicts **velocity** $v$.

For a straight-line path $x_\tau = (1-\tau) x_0 + \tau x_1$, the velocity is:

$$v = \frac{dx_\tau}{d\tau} = x_1 - x_0$$

This is a **constant** — the velocity doesn't change along the path! It's just the direction from noise to data.

| | DDPM | Flow Matching |
|---|---|---|
| **Predicts** | Noise $\epsilon$ | Velocity $v = x_1 - x_0$ |
| **Path** | Curved (SDE) | Straight (ODE) |
| **Schedule** | Complex ($\beta_t$, $\bar{\alpha}_t$) | None (just $\tau \in [0,1]$) |
| **Time** | Discrete $t \in \{0,...,T\}$ | Continuous $\tau \in [0,1]$ |

In [ ]:
class VelocityNet(nn.Module):
    """Predict velocity v given (x_tau, tau).

    Architecture: sinusoidal time embedding + 3-layer MLP.
    """
    def __init__(self, data_dim=2, hidden=256, time_dim=64):
        super().__init__()
        self.time_dim = time_dim

        # Time embedding: tau -> [sin, cos] features -> hidden
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, hidden),
            nn.SiLU(),
        )

        # Main network: (x concat time_features) -> velocity
        self.net = nn.Sequential(
            nn.Linear(data_dim + hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, data_dim),
        )

    def sinusoidal_embed(self, tau):
        """Sinusoidal time embedding: tau -> [sin, cos] features."""
        half = self.time_dim // 2
        freqs = torch.exp(
            -math.log(10000.0) * torch.arange(half, device=tau.device).float() / half
        )
        args = tau * freqs.unsqueeze(0)          # [batch, half]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # [batch, time_dim]

    def forward(self, x, tau):
        """x: [batch, data_dim], tau: [batch, 1] -> velocity [batch, data_dim]"""
        t_emb = self.sinusoidal_embed(tau)        # [batch, time_dim]
        t_feat = self.time_mlp(t_emb)             # [batch, hidden]
        h = torch.cat([x, t_feat], dim=-1)        # [batch, data_dim + hidden]
        return self.net(h)                         # [batch, data_dim]


# Test the network
vel_net = VelocityNet(data_dim=2, hidden=256).to(device)
test_x = torch.randn(4, 2, device=device)
test_tau = torch.rand(4, 1, device=device)

with torch.no_grad():
    test_v = vel_net(test_x, test_tau)

n_params = sum(p.numel() for p in vel_net.parameters())
print(f'VelocityNet:')
print(f'  Input: x {list(test_x.shape)} + tau {list(test_tau.shape)}')
print(f'  Output: velocity {list(test_v.shape)}')
print(f'  Parameters: {n_params:,}')
print(f'\nKey: predicts a 2D velocity vector at each point in space-time.')
print(f'At inference, we follow these velocity vectors from noise to data.')

---
## Step 5: The Flow Matching Training Loop

The training loop is remarkably simple — just **15 lines** of core logic:

1. Sample $\tau \sim \text{Uniform}[0, 1]$
2. Sample $x_0 \sim \mathcal{N}(0, I)$ (noise)
3. Sample $x_1$ from the dataset (data)
4. Interpolate: $x_\tau = (1 - \tau) x_0 + \tau x_1$
5. Target velocity: $v^* = x_1 - x_0$
6. Predicted velocity: $\hat{v} = \text{model}(x_\tau, \tau)$
7. Loss: $\mathcal{L} = \| \hat{v} - v^* \|^2$

That's it. No noise schedule. No complex math. Just predict the straight-line direction.

In [ ]:
# ── Flow Matching Training Loop ─────────────────────────────────
model = VelocityNet(data_dim=2, hidden=256).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
data_device = data.to(device)

fm_losses = []
model.train()

for step in range(2000):
    # ── Core: 15 lines ──────────────────────────────────
    batch_size = 256

    # 1. Sample tau ~ Uniform[0, 1]
    tau = torch.rand(batch_size, 1, device=device)

    # 2. Sample x0 ~ N(0, I)  (noise)
    x0 = torch.randn(batch_size, 2, device=device)

    # 3. Sample x1 from dataset (data)
    idx = torch.randint(0, len(data), (batch_size,))
    x1 = data_device[idx]

    # 4. Interpolate: x_tau = (1 - tau) * x0 + tau * x1
    x_tau = (1 - tau) * x0 + tau * x1

    # 5. Target velocity: v* = x1 - x0
    target = x1 - x0

    # 6. Predicted velocity: v_hat = model(x_tau, tau)
    predicted = model(x_tau, tau)

    # 7. Loss: MSE(v_hat, v*)
    loss = F.mse_loss(predicted, target)
    # ────────────────────────────────────────────────────

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    fm_losses.append(loss.item())

    if step % 500 == 0:
        print(f'  Step {step:4d}: loss = {loss.item():.4f}')

print(f'\nFlow Matching training done. Final loss: {fm_losses[-1]:.4f}')

In [ ]:
# Plot: loss curve
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(fm_losses, color=TEAL, alpha=0.3, linewidth=0.5)
# Smoothed
window = 50
smoothed = np.convolve(fm_losses, np.ones(window)/window, mode='valid')
ax.plot(range(window-1, len(fm_losses)), smoothed, color=TEAL, linewidth=2.5)
ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('Flow Matching Training Loss', fontweight='bold', color=TEAL, fontsize=14)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()
print(f'Loss dropped from {fm_losses[0]:.3f} to {fm_losses[-1]:.3f}')

In [ ]:
# Velocity field quiver plots at different tau values
tau_values = [0.0, 0.25, 0.5, 0.75, 1.0]

fig, axes = plt.subplots(1, 5, figsize=(25, 5))

# Create grid for velocity field
grid_n = 15
gx = torch.linspace(-3, 3, grid_n)
gy = torch.linspace(-3, 3, grid_n)
grid_x, grid_y = torch.meshgrid(gx, gy, indexing='ij')
grid_pts = torch.stack([grid_x.flatten(), grid_y.flatten()], dim=-1).to(device)  # [225, 2]

model.eval()
for i, tau_val in enumerate(tau_values):
    tau_batch = torch.full((grid_pts.shape[0], 1), tau_val, device=device)
    with torch.no_grad():
        vel = model(grid_pts, tau_batch).cpu().numpy()

    gx_np = grid_pts[:, 0].cpu().numpy()
    gy_np = grid_pts[:, 1].cpu().numpy()

    ax = axes[i]
    # Show data distribution faintly
    ax.scatter(data[:500, 0].numpy(), data[:500, 1].numpy(), s=2, alpha=0.1, c='gray')
    # Quiver plot
    ax.quiver(gx_np, gy_np, vel[:, 0], vel[:, 1],
              color=TEAL, alpha=0.7, scale=60, width=0.005)
    ax.set_title(f'$\\tau$ = {tau_val:.2f}', fontweight='bold', color=TEAL, fontsize=13)
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)

plt.suptitle('Learned Velocity Field at Different Times',
             fontsize=16, fontweight='bold', color=TEAL)
plt.tight_layout()
plt.show()
print('At tau=0: velocities point from noise cloud TOWARD the data manifold.')
print('At tau=1: velocities are small — samples have already arrived at the data.')
print('The velocity field smoothly interpolates between these extremes.')

---
## Step 6: Inference — Euler Integration

To generate samples, we follow the learned velocity field from $\tau=0$ (noise) to $\tau=1$ (data) using Euler integration:

$$x_{\tau + \Delta\tau} = x_\tau + \Delta\tau \cdot v_\theta(x_\tau, \tau)$$

Because the paths are straight, even **10 steps** gives good quality!

In [ ]:
@torch.no_grad()
def euler_integrate(model, n_steps=10, n_samples=500):
    """Generate samples by Euler integration of the velocity field."""
    model.eval()
    x = torch.randn(n_samples, 2, device=device)
    dt = 1.0 / n_steps
    trajectory = [x.cpu().numpy().copy()]

    for i in range(n_steps):
        tau = torch.full((n_samples, 1), i * dt, device=device)
        v = model(x, tau)
        x = x + dt * v
        trajectory.append(x.cpu().numpy().copy())

    return x.cpu(), np.array(trajectory)


# Generate with 10 steps
samples_10, traj_10 = euler_integrate(model, n_steps=10, n_samples=500)
print(f'Generated {samples_10.shape[0]} samples in 10 Euler steps')
print(f'  Sample mean: [{samples_10[:, 0].mean():.2f}, {samples_10[:, 1].mean():.2f}]')
print(f'  Sample std:  [{samples_10[:, 0].std():.2f}, {samples_10[:, 1].std():.2f}]')

In [ ]:
# 5-panel figure showing quality at different step counts
step_options = [1, 5, 10, 20, 50]
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
panel_colors = [ACCENT, WARM, TEAL, BLUE, PURPLE]

for i, n_steps in enumerate(step_options):
    samples, _ = euler_integrate(model, n_steps=n_steps, n_samples=500)
    ax = axes[i]
    # Reference data
    ax.scatter(data[:500, 0].numpy(), data[:500, 1].numpy(), s=3, alpha=0.15, c='gray',
               label='Real data')
    # Generated samples
    ax.scatter(samples[:, 0].numpy(), samples[:, 1].numpy(), s=4, alpha=0.5,
               c=panel_colors[i], label='Generated')
    ax.set_title(f'{n_steps} step{"s" if n_steps > 1 else ""}',
                 fontweight='bold', color=panel_colors[i], fontsize=14)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    if i == 0:
        ax.legend(fontsize=8, loc='upper left')

plt.suptitle('Flow Matching: Sample Quality vs Number of Euler Steps',
             fontsize=16, fontweight='bold', color=TEAL)
plt.tight_layout()
plt.show()
print('1 step:  Just one big jump — rough approximation.')
print('5 steps: Already recognizable as Swiss roll!')
print('10+ steps: High quality. Compare to DDPM needing 100+ steps.')

In [ ]:
# Compute MMD (Maximum Mean Discrepancy) between generated and real data
def compute_mmd(x, y, sigma=1.0):
    """Gaussian kernel MMD between two sets of samples."""
    xx = torch.cdist(x, x)
    yy = torch.cdist(y, y)
    xy = torch.cdist(x, y)
    kxx = torch.exp(-xx**2 / (2 * sigma**2)).mean()
    kyy = torch.exp(-yy**2 / (2 * sigma**2)).mean()
    kxy = torch.exp(-xy**2 / (2 * sigma**2)).mean()
    return (kxx + kyy - 2 * kxy).item()


# Compute MMD at different step counts
step_range = [1, 2, 3, 5, 8, 10, 15, 20, 30, 50]
mmd_values = []
real_subset = data[:500]

for n_steps in step_range:
    samples, _ = euler_integrate(model, n_steps=n_steps, n_samples=500)
    mmd = compute_mmd(samples, real_subset, sigma=1.0)
    mmd_values.append(mmd)
    print(f'  {n_steps:3d} steps: MMD = {mmd:.6f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(step_range, mmd_values, 'o-', color=TEAL, linewidth=2.5, markersize=8)
ax.set_xlabel('Number of Euler Steps', fontsize=12)
ax.set_ylabel('MMD (lower = better)', fontsize=12)
ax.set_title('Sample Quality vs Euler Steps (MMD)',
             fontweight='bold', color=TEAL, fontsize=14)
ax.grid(True, alpha=0.2)
# Annotate the "sweet spot"
best_idx = np.argmin(mmd_values)
ax.annotate(f'Best: {step_range[best_idx]} steps\nMMD={mmd_values[best_idx]:.5f}',
            xy=(step_range[best_idx], mmd_values[best_idx]),
            xytext=(step_range[best_idx] + 10, mmd_values[best_idx] + 0.001),
            fontsize=11, color=ACCENT, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=ACCENT, lw=1.5))
plt.tight_layout()
plt.show()
print(f'\nMMD converges quickly — 10 steps is already near-optimal.')
print(f'This is 10-100x fewer steps than DDPM!')

---
## Step 7: Conditional Flow Matching + Optimal Transport Coupling

So far we **randomly pair** noise samples $x_0$ with data points $x_1$. This creates **crossing paths** — two nearby noise points might be paired with data points on opposite sides.

**Optimal Transport (OT)** fixes this by finding the **best pairing** that minimizes total path length:

$$\min_{\pi} \sum_{(i,j) \in \pi} \| x_0^{(i)} - x_1^{(j)} \|^2$$

This gives **uncrossed, shorter paths** — easier for the network to learn.

In [ ]:
# Optimal Transport pairing using scipy's linear_sum_assignment
def ot_pairing(x0, x1):
    """Find optimal transport pairing between x0 and x1.
    Returns reordered x0 that minimizes total squared distance to x1."""
    cost = torch.cdist(x0, x1).cpu().numpy() ** 2  # [n, n] squared distances
    row_idx, col_idx = linear_sum_assignment(cost)
    # Reorder x0 so that x0[i] is paired with x1[i]
    # We reorder x0 to match x1's ordering
    x0_reordered = x0[row_idx[np.argsort(col_idx)]]
    return x0_reordered


# Visualize: Random vs OT pairing
n_viz = 30
x0_viz = torch.randn(n_viz, 2)
idx_viz = torch.randint(0, len(data), (n_viz,))
x1_viz = data[idx_viz]
x0_ot = ot_pairing(x0_viz, x1_viz)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Random pairing
ax = axes[0]
for i in range(n_viz):
    ax.plot([x0_viz[i, 0], x1_viz[i, 0]], [x0_viz[i, 1], x1_viz[i, 1]],
            '-', color=ACCENT, alpha=0.4, linewidth=1)
ax.scatter(x0_viz[:, 0], x0_viz[:, 1], s=30, c=BLUE, zorder=5,
           edgecolors='white', linewidths=0.5, label='Noise (x0)')
ax.scatter(x1_viz[:, 0], x1_viz[:, 1], s=30, c=TEAL, zorder=5,
           edgecolors='white', linewidths=0.5, label='Data (x1)')
ax.set_title('Random Pairing: Crossed Paths', fontweight='bold', color=ACCENT, fontsize=14)
ax.legend(fontsize=10)
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)

# OT pairing
ax = axes[1]
for i in range(n_viz):
    ax.plot([x0_ot[i, 0], x1_viz[i, 0]], [x0_ot[i, 1], x1_viz[i, 1]],
            '-', color=TEAL, alpha=0.4, linewidth=1)
ax.scatter(x0_ot[:, 0], x0_ot[:, 1], s=30, c=BLUE, zorder=5,
           edgecolors='white', linewidths=0.5, label='Noise (x0, reordered)')
ax.scatter(x1_viz[:, 0], x1_viz[:, 1], s=30, c=TEAL, zorder=5,
           edgecolors='white', linewidths=0.5, label='Data (x1)')
ax.set_title('OT Pairing: Uncrossed Paths', fontweight='bold', color=TEAL, fontsize=14)
ax.legend(fontsize=10)
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)

plt.suptitle('Random vs Optimal Transport Coupling',
             fontsize=16, fontweight='bold', color=ACCENT)
plt.tight_layout()
plt.show()

# Compute total path lengths
random_len = torch.norm(x0_viz - x1_viz, dim=-1).sum().item()
ot_len = torch.norm(x0_ot - x1_viz, dim=-1).sum().item()
print(f'Total path length (random): {random_len:.2f}')
print(f'Total path length (OT):     {ot_len:.2f}')
print(f'OT is {random_len / ot_len:.1f}x shorter!')
print('\nFewer crossings = simpler velocity field = easier for the network to learn.')

In [ ]:
# Train both variants: random coupling vs OT coupling
def train_flow_matching(use_ot=False, n_steps=2000, batch_size=256):
    """Train flow matching with optional OT coupling."""
    net = VelocityNet(data_dim=2, hidden=256).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    losses = []

    net.train()
    for step in range(n_steps):
        tau = torch.rand(batch_size, 1, device=device)
        x0 = torch.randn(batch_size, 2, device=device)
        idx = torch.randint(0, len(data), (batch_size,))
        x1 = data_device[idx]

        if use_ot:
            # OT mini-batch: reorder x0 to minimize transport cost
            x0 = ot_pairing(x0, x1)

        x_tau = (1 - tau) * x0 + tau * x1
        target = x1 - x0
        predicted = net(x_tau, tau)
        loss = F.mse_loss(predicted, target)

        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())

    return net, losses


print('Training with random coupling...')
model_random, losses_random = train_flow_matching(use_ot=False, n_steps=2000)
print(f'  Final loss: {losses_random[-1]:.4f}')

# OT with smaller batches (scipy linear_sum_assignment is O(n^3))
print('Training with OT coupling (batch=64 for OT speed)...')
model_ot, losses_ot = train_flow_matching(use_ot=True, n_steps=2000, batch_size=64)
print(f'  Final loss: {losses_ot[-1]:.4f}')

In [ ]:
# Compare loss curves
fig, ax = plt.subplots(figsize=(10, 5))

window = 50
sm_random = np.convolve(losses_random, np.ones(window)/window, mode='valid')
sm_ot = np.convolve(losses_ot, np.ones(window)/window, mode='valid')

ax.plot(range(window-1, len(losses_random)), sm_random, color=ACCENT, linewidth=2.5,
        label='Random coupling')
ax.plot(range(window-1, len(losses_ot)), sm_ot, color=TEAL, linewidth=2.5,
        label='OT coupling')
ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('MSE Loss (smoothed)', fontsize=12)
ax.set_title('Random vs OT Coupling: Training Loss',
             fontweight='bold', color=ACCENT, fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print(f'Random coupling final loss: {np.mean(losses_random[-100:]):.4f}')
print(f'OT coupling final loss:     {np.mean(losses_ot[-100:]):.4f}')
print('\nOT coupling typically achieves lower loss — the velocity field is simpler to learn.')

---
## Step 8: Flow Matching for Robot Actions

Now let's apply flow matching to something closer to robotics: **generating action trajectories**.

Setup:
- 4 push directions: up, down, left, right
- Each action is an **8-step 2D trajectory** (8 waypoints in xy space)
- The model is **conditioned** on a one-hot direction vector
- At inference: given a direction, generate a smooth push trajectory

This mirrors how real robot policies work: given an observation, generate an action sequence.

In [ ]:
# Create synthetic expert push trajectories
def create_push_data(n_per_direction=200, n_waypoints=8):
    """Create smooth 8-step push trajectories in 4 directions."""
    trajectories = []
    directions = []

    direction_vecs = {
        0: np.array([0.0, 1.0]),   # up
        1: np.array([0.0, -1.0]),  # down
        2: np.array([-1.0, 0.0]),  # left
        3: np.array([1.0, 0.0]),   # right
    }

    for dir_idx, dir_vec in direction_vecs.items():
        for _ in range(n_per_direction):
            # Start near origin with slight random offset
            start = np.random.randn(2) * 0.1
            # Speed variation
            speed = 0.3 + np.random.rand() * 0.15
            # Slight angle jitter
            angle_jitter = np.random.randn() * 0.15
            rot = np.array([[np.cos(angle_jitter), -np.sin(angle_jitter)],
                            [np.sin(angle_jitter), np.cos(angle_jitter)]])
            actual_dir = rot @ dir_vec

            # Create smooth trajectory
            traj = []
            pos = start.copy()
            for t in range(n_waypoints):
                # Smooth acceleration/deceleration
                progress = (t + 1) / n_waypoints
                step_speed = speed * np.sin(progress * np.pi)  # sine curve
                pos = pos + actual_dir * step_speed * 0.15
                # Small noise
                pos = pos + np.random.randn(2) * 0.01
                traj.append(pos.copy())

            trajectories.append(np.array(traj).flatten())  # [16] = 8 waypoints * 2d
            directions.append(dir_idx)

    return (torch.tensor(np.array(trajectories), dtype=torch.float32),
            torch.tensor(directions, dtype=torch.long))


traj_data, dir_labels = create_push_data(n_per_direction=200)
print(f'Trajectory data: {list(traj_data.shape)} (800 trajectories, 16-d = 8 waypoints * 2d)')
print(f'Direction labels: {list(dir_labels.shape)} (0=up, 1=down, 2=left, 3=right)')
print(f'  Per direction: {[(dir_labels == i).sum().item() for i in range(4)]}')

In [ ]:
# Visualize expert trajectories
dir_names = ['Up', 'Down', 'Left', 'Right']
dir_colors = [ACCENT, BLUE, TEAL, PURPLE]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for d in range(4):
    ax = axes[d]
    mask = dir_labels == d
    trajs = traj_data[mask].numpy()

    for i in range(min(30, len(trajs))):
        traj = trajs[i].reshape(8, 2)
        ax.plot(traj[:, 0], traj[:, 1], '-', color=dir_colors[d], alpha=0.3, linewidth=1.5)
        ax.scatter(traj[-1, 0], traj[-1, 1], s=15, color=dir_colors[d], alpha=0.3)

    ax.scatter([0], [0], marker='x', s=100, color='black', zorder=10, linewidths=2)
    ax.set_title(f'{dir_names[d]}', fontweight='bold', color=dir_colors[d], fontsize=14)
    ax.set_xlim(-0.8, 0.8)
    ax.set_ylim(-0.8, 0.8)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)

plt.suptitle('Expert Push Trajectories (4 Directions)',
             fontsize=16, fontweight='bold', color=ACCENT)
plt.tight_layout()
plt.show()
print('Each trajectory: 8 waypoints showing how to push in one direction.')
print('The model must learn to generate these given just the direction label.')

In [ ]:
class ConditionedVelocityNet(nn.Module):
    """Velocity network conditioned on observation (one-hot direction).

    Takes: (x_tau, tau, condition) -> velocity
    """
    def __init__(self, data_dim=16, cond_dim=4, hidden=256, time_dim=64):
        super().__init__()
        self.time_dim = time_dim

        # Time embedding
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, hidden),
            nn.SiLU(),
        )

        # Condition embedding
        self.cond_mlp = nn.Sequential(
            nn.Linear(cond_dim, hidden),
            nn.SiLU(),
        )

        # Main network: (x + time_feat + cond_feat) -> velocity
        self.net = nn.Sequential(
            nn.Linear(data_dim + hidden + hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, data_dim),
        )

    def sinusoidal_embed(self, tau):
        half = self.time_dim // 2
        freqs = torch.exp(
            -math.log(10000.0) * torch.arange(half, device=tau.device).float() / half
        )
        args = tau * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

    def forward(self, x, tau, cond):
        """x: [B, data_dim], tau: [B, 1], cond: [B, cond_dim] -> [B, data_dim]"""
        t_emb = self.sinusoidal_embed(tau)
        t_feat = self.time_mlp(t_emb)
        c_feat = self.cond_mlp(cond)
        h = torch.cat([x, t_feat, c_feat], dim=-1)
        return self.net(h)


cond_model = ConditionedVelocityNet(data_dim=16, cond_dim=4, hidden=256).to(device)
n_params = sum(p.numel() for p in cond_model.parameters())
print(f'ConditionedVelocityNet:')
print(f'  data_dim=16 (8 waypoints * 2d), cond_dim=4 (one-hot direction)')
print(f'  Parameters: {n_params:,}')

In [ ]:
# Train conditioned flow matching on push trajectories
traj_device = traj_data.to(device)
dir_device = dir_labels.to(device)

optimizer = torch.optim.Adam(cond_model.parameters(), lr=1e-3)
cond_losses = []
cond_model.train()

for step in range(3000):
    batch_size = 128
    idx = torch.randint(0, len(traj_data), (batch_size,))

    x1 = traj_device[idx]                              # [B, 16]
    cond = F.one_hot(dir_device[idx], 4).float()       # [B, 4]

    tau = torch.rand(batch_size, 1, device=device)
    x0 = torch.randn(batch_size, 16, device=device)

    x_tau = (1 - tau) * x0 + tau * x1
    target = x1 - x0
    predicted = cond_model(x_tau, tau, cond)

    loss = F.mse_loss(predicted, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    cond_losses.append(loss.item())

    if step % 1000 == 0:
        print(f'  Step {step:4d}: loss = {loss.item():.4f}')

print(f'\nConditioned training done. Final loss: {cond_losses[-1]:.4f}')

In [ ]:
# Generate conditioned trajectories and visualize
@torch.no_grad()
def generate_trajectories(model, direction_idx, n_samples=30, n_steps=20):
    """Generate trajectories conditioned on direction."""
    model.eval()
    x = torch.randn(n_samples, 16, device=device)
    cond = F.one_hot(torch.full((n_samples,), direction_idx, dtype=torch.long),
                     4).float().to(device)
    dt = 1.0 / n_steps

    for i in range(n_steps):
        tau = torch.full((n_samples, 1), i * dt, device=device)
        v = model(x, tau, cond)
        x = x + dt * v

    return x.cpu().numpy().reshape(n_samples, 8, 2)


fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for d in range(4):
    ax = axes[d]

    # Show expert trajectories (faded)
    mask = dir_labels == d
    expert_trajs = traj_data[mask].numpy()
    for i in range(min(20, len(expert_trajs))):
        traj = expert_trajs[i].reshape(8, 2)
        ax.plot(traj[:, 0], traj[:, 1], '-', color='gray', alpha=0.15, linewidth=1)

    # Show generated trajectories
    gen_trajs = generate_trajectories(cond_model, d, n_samples=30, n_steps=20)
    for i in range(len(gen_trajs)):
        traj = gen_trajs[i]
        ax.plot(traj[:, 0], traj[:, 1], '-', color=dir_colors[d], alpha=0.5, linewidth=1.5)
        ax.scatter(traj[-1, 0], traj[-1, 1], s=20, color=dir_colors[d], alpha=0.5)

    ax.scatter([0], [0], marker='x', s=100, color='black', zorder=10, linewidths=2)
    ax.set_title(f'{dir_names[d]} (Generated)',
                 fontweight='bold', color=dir_colors[d], fontsize=14)
    ax.set_xlim(-0.8, 0.8)
    ax.set_ylim(-0.8, 0.8)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)

plt.suptitle('Flow Matching: Generated Push Trajectories (colored) vs Expert (gray)',
             fontsize=15, fontweight='bold', color=ACCENT)
plt.tight_layout()
plt.show()
print('Generated trajectories (colored) match expert distribution (gray)!')
print('The model learned to produce different trajectories for each direction.')
print('\nThis is exactly how pi0 generates robot actions:')
print('  Observation (image + text) -> conditioned flow matching -> action trajectory')

---
## Step 9: DDPM vs Flow Matching — Head-to-Head

Let's directly compare both methods on the **same Swiss roll data** across three metrics:
1. Inference speed
2. Sample quality (MMD)
3. Convergence speed (loss at different training steps)

In [ ]:
# ── Retrain both from scratch for fair comparison ────────────────
print('Training DDPM (2000 steps)...')
ddpm_compare = SimpleDDPM(T=100)
ddpm_net_compare = DDPMNoiseNet(data_dim=2, hidden=256).to(device)
opt_ddpm = torch.optim.Adam(ddpm_net_compare.parameters(), lr=1e-3)
ddpm_compare_losses = []

ddpm_net_compare.train()
for step in range(2000):
    idx = torch.randint(0, len(data), (256,))
    x0 = data_device[idx]
    t = torch.randint(0, ddpm_compare.T, (256,), device=device)
    eps = torch.randn_like(x0)
    sqrt_ab = ddpm_compare.sqrt_alpha_bar[t.cpu()].unsqueeze(-1).to(device)
    sqrt_omab = ddpm_compare.sqrt_one_minus_alpha_bar[t.cpu()].unsqueeze(-1).to(device)
    x_t = sqrt_ab * x0 + sqrt_omab * eps
    eps_pred = ddpm_net_compare(x_t, t)
    loss = F.mse_loss(eps_pred, eps)
    opt_ddpm.zero_grad()
    loss.backward()
    opt_ddpm.step()
    ddpm_compare_losses.append(loss.item())
print(f'  DDPM final loss: {ddpm_compare_losses[-1]:.4f}')

print('Training Flow Matching (2000 steps)...')
fm_net_compare = VelocityNet(data_dim=2, hidden=256).to(device)
opt_fm = torch.optim.Adam(fm_net_compare.parameters(), lr=1e-3)
fm_compare_losses = []

fm_net_compare.train()
for step in range(2000):
    tau = torch.rand(256, 1, device=device)
    x0 = torch.randn(256, 2, device=device)
    idx = torch.randint(0, len(data), (256,))
    x1 = data_device[idx]
    x_tau = (1 - tau) * x0 + tau * x1
    target = x1 - x0
    predicted = fm_net_compare(x_tau, tau)
    loss = F.mse_loss(predicted, target)
    opt_fm.zero_grad()
    loss.backward()
    opt_fm.step()
    fm_compare_losses.append(loss.item())
print(f'  FM final loss: {fm_compare_losses[-1]:.4f}')

In [ ]:
# ── Comparison 1: Inference time ────────────────────────────────
ddpm_inf_steps = [50, 100]
fm_inf_steps = [5, 10]

ddpm_times_compare = []
fm_times_compare = []

for T in ddpm_inf_steps:
    ddpm_v = SimpleDDPM(T=T)
    _ = ddpm_sample(ddpm_v, ddpm_net_compare, n_samples=100, T=T)
    start = time.perf_counter()
    for _ in range(5):
        _ = ddpm_sample(ddpm_v, ddpm_net_compare, n_samples=500, T=T)
    ddpm_times_compare.append((time.perf_counter() - start) / 5 * 1000)

for ns in fm_inf_steps:
    _ = euler_integrate(fm_net_compare, n_steps=ns, n_samples=100)
    start = time.perf_counter()
    for _ in range(5):
        _ = euler_integrate(fm_net_compare, n_steps=ns, n_samples=500)
    fm_times_compare.append((time.perf_counter() - start) / 5 * 1000)

# ── Comparison 2: Sample quality (MMD) ──────────────────────────
real_subset = data[:500]

ddpm_v100 = SimpleDDPM(T=100)
ddpm_samples_100 = ddpm_sample(ddpm_v100, ddpm_net_compare, n_samples=500, T=100).cpu()
ddpm_mmd = compute_mmd(ddpm_samples_100, real_subset)

fm_samples_10, _ = euler_integrate(fm_net_compare, n_steps=10, n_samples=500)
fm_mmd = compute_mmd(fm_samples_10, real_subset)

print(f'MMD comparison:')
print(f'  DDPM (100 steps): {ddpm_mmd:.6f}')
print(f'  FM   (10 steps):  {fm_mmd:.6f}')

In [ ]:
# ── Three-panel comparison figure ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 5))

# Panel 1: Inference time
ax = axes[0]
labels = [f'DDPM\n{s} steps' for s in ddpm_inf_steps] + [f'FM\n{s} steps' for s in fm_inf_steps]
times_all = ddpm_times_compare + fm_times_compare
bar_colors = [ACCENT, ACCENT, TEAL, TEAL]
bars = ax.bar(labels, times_all, color=bar_colors, edgecolor='white', linewidth=1.5)
for bar, t_ms in zip(bars, times_all):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{t_ms:.0f} ms', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Inference Time (ms)', fontsize=12)
ax.set_title('Inference Speed', fontweight='bold', color=ACCENT, fontsize=14)
ax.grid(True, alpha=0.2, axis='y')

# Panel 2: Sample quality (scatter)
ax = axes[1]
ax.scatter(ddpm_samples_100[:, 0].numpy(), ddpm_samples_100[:, 1].numpy(),
           s=4, alpha=0.4, c=ACCENT, label=f'DDPM (MMD={ddpm_mmd:.5f})')
ax.scatter(fm_samples_10[:, 0].numpy(), fm_samples_10[:, 1].numpy(),
           s=4, alpha=0.4, c=TEAL, label=f'FM (MMD={fm_mmd:.5f})')
ax.scatter(data[:300, 0].numpy(), data[:300, 1].numpy(),
           s=2, alpha=0.1, c='gray', label='Real')
ax.set_title('Sample Quality', fontweight='bold', color=ACCENT, fontsize=14)
ax.set_aspect('equal')
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.2)

# Panel 3: Convergence (loss curves)
ax = axes[2]
window = 50
sm_ddpm = np.convolve(ddpm_compare_losses, np.ones(window)/window, mode='valid')
sm_fm = np.convolve(fm_compare_losses, np.ones(window)/window, mode='valid')
ax.plot(range(window-1, len(ddpm_compare_losses)), sm_ddpm, color=ACCENT,
        linewidth=2.5, label='DDPM')
ax.plot(range(window-1, len(fm_compare_losses)), sm_fm, color=TEAL,
        linewidth=2.5, label='Flow Matching')
ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('Convergence', fontweight='bold', color=ACCENT, fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.2)

plt.suptitle('DDPM vs Flow Matching: Complete Comparison',
             fontsize=16, fontweight='bold', color=ACCENT)
plt.tight_layout()
plt.show()

In [ ]:
# ── Convergence at specific checkpoints ─────────────────────────
checkpoints = [500, 1000, 2000]
print('\n' + '='*60)
print('FINAL COMPARISON TABLE')
print('='*60)
print(f'{"Metric":<30} {"DDPM":>12} {"Flow Matching":>15}')
print('-'*60)

for cp in checkpoints:
    ddpm_loss_cp = np.mean(ddpm_compare_losses[max(0,cp-50):cp])
    fm_loss_cp = np.mean(fm_compare_losses[max(0,cp-50):cp])
    print(f'Loss @ step {cp:<18} {ddpm_loss_cp:>12.4f} {fm_loss_cp:>15.4f}')

print(f'{"MMD (sample quality)":<30} {ddpm_mmd:>12.6f} {fm_mmd:>15.6f}')
print(f'{"Inference steps":<30} {"100":>12} {"10":>15}')
print(f'{"Inference time (ms)":<30} {ddpm_times_compare[1]:>12.0f} {fm_times_compare[1]:>15.0f}')
print(f'{"Noise schedule needed":<30} {"Yes":>12} {"No":>15}')
print(f'{"Path geometry":<30} {"Curved":>12} {"Straight":>15}')
print('='*60)
print('\nFlow matching wins on speed and simplicity.')
print('This is why pi0 and modern robot policies use flow matching over DDPM.')

---

## Summary

| | DDPM | Flow Matching |
|---|---|---|
| **Paths** | Curved, stochastic (SDE) | Straight lines (ODE) |
| **Schedule** | Complex ($\beta_t$, $\bar{\alpha}_t$, 7+ precomputed tensors) | None ($\tau \in [0,1]$) |
| **Predicts** | Noise $\epsilon$ | Velocity $v = x_1 - x_0$ |
| **Training** | ~30 lines | **15 lines** |
| **Inference** | 50-1000 steps | **5-10 steps** |
| **OT coupling** | Not standard | Natural extension, reduces crossings |
| **Used by** | DDPM, Stable Diffusion v1 | **pi0, Stable Diffusion 3, Flux** |

**Key takeaway:** Flow matching replaces DDPM's complex forward/reverse process with simple linear interpolation and velocity prediction. The straight-line paths enable fast inference (10x fewer steps) with comparable quality.

In **Part 2**, we'll build the **DiT (Diffusion Transformer)** architecture that sits at the heart of pi0's action expert.

---

## Exercises

1. **Euler step count ablation:** Plot MMD vs number of Euler steps from 1 to 100. Where is the diminishing returns threshold? Does it match the visual quality from the 5-panel figure?

2. **Heun's method (2nd-order Runge-Kutta):** Implement Heun's method for integration: $k_1 = v(x, \tau)$, $k_2 = v(x + dt \cdot k_1, \tau + dt)$, $x_{\text{next}} = x + \frac{dt}{2}(k_1 + k_2)$. Compare with Euler at 5 steps — does Heun match Euler at 10 steps?

3. **Noise distribution:** Replace $x_0 \sim \mathcal{N}(0, I)$ with $x_0 \sim \text{Uniform}(-3, 3)$. Retrain and compare sample quality. Does the source distribution matter much?

4. **Time distribution:** Instead of sampling $\tau \sim \text{Uniform}[0, 1]$, try logit-normal sampling: $\tau = \sigma(z)$ where $z \sim \mathcal{N}(0, 1)$. This biases training toward $\tau \approx 0.5$. Does it help convergence?

5. **Scale to 7D actions:** Modify `ConditionedVelocityNet` to handle 7D actions (6 joint angles + 1 gripper) with 8-step trajectories (data_dim=56). Create synthetic data and verify training works.

6. **Challenge:** Replace `VelocityNet` (MLP) with a **DiT-style transformer** as the velocity network. Use self-attention over the data dimensions with AdaLN conditioning on $\tau$. Compare training speed and sample quality vs the MLP baseline.